# CMU 10-405/605/805 Machine Learning with Large Datasets

## Homework 3 — Coding: Distributed Training

## Part A: Correctness tests for student code

Tests 1–4 check the four TODO functions in `train_distributed.py` using small inputs.


In [ ]:
import torch
import torch.distributed as dist
from torch.utils.data import DistributedSampler, TensorDataset
from unittest.mock import patch
from train_distributed import (MLP, initialize_distributed,
                               broadcast_parameters, average_gradients, average_model)


### Test 1 — Initialize the process group


In [ ]:
# PUBLIC TESTS — test_initialize_distributed
with patch.object(dist, "init_process_group") as start, \
     patch.object(dist, "get_rank", return_value=1), \
     patch.object(dist, "get_world_size", return_value=2):
    rank, world_size = initialize_distributed()
    assert start.call_count == 1
    assert start.call_args.kwargs.get("backend") == "gloo"
    assert rank == 1
    assert world_size == 2


### Test 2 — Broadcast model parameters


In [ ]:
# PUBLIC TESTS — test_broadcast_parameters
model = MLP(80)
parameters = list(model.parameters())
with torch.no_grad():
    for parameter in parameters:
        parameter.fill_(-1)

seen = []
def simulate_broadcast(tensor, src):
    assert src == 0
    with torch.no_grad():
        tensor.fill_(3 + len(seen))  # Distinct values sent by rank 0.
    seen.append(tensor)

with patch.object(dist, "broadcast", side_effect=simulate_broadcast):
    broadcast_parameters(model)
assert len(seen) == len(parameters)
assert all(received is parameter for received, parameter in zip(seen, parameters))
assert all(torch.all(parameter == 3 + i) for i, parameter in enumerate(parameters))


### Test 3 — Average packed gradients


In [ ]:
# PUBLIC TESTS — test_average_gradients
model = MLP(80)
parameters = list(model.parameters())
for i, parameter in enumerate(parameters):
    parameter.grad = torch.full_like(parameter, float(i + 1))
shapes = [parameter.grad.shape for parameter in parameters]
other_worker = torch.cat([torch.full_like(parameter.grad, float(i + 3)).flatten()
                          for i, parameter in enumerate(parameters)])
local_worker = torch.cat([parameter.grad.flatten() for parameter in parameters])

def simulate_all_reduce(packed, *args, **kwargs):
    operation = kwargs.get("op", args[0] if args else dist.ReduceOp.SUM)
    assert operation == dist.ReduceOp.SUM
    assert packed.ndim == 1 and packed.numel() == other_worker.numel()
    assert torch.allclose(packed, local_worker)
    packed.add_(other_worker)  # Simulate a SUM from two workers.

with patch.object(dist, "all_reduce", side_effect=simulate_all_reduce) as collective:
    average_gradients(model, world_size=2)
    assert collective.call_count == 1  # All gradients must be packed together.
    assert all(parameter.grad.shape == shape for parameter, shape in zip(parameters, shapes))
    assert all(torch.allclose(parameter.grad, torch.full_like(parameter, i + 2))
               for i, parameter in enumerate(parameters))
    average_gradients(model, world_size=1)
    assert collective.call_count == 1  # A single worker needs no communication.
    assert all(torch.allclose(parameter.grad, torch.full_like(parameter, i + 2))
               for i, parameter in enumerate(parameters))


### Test 4 -- Average model parameters

In [ ]:
model = MLP(80)
parameters = list(model.parameters())
with torch.no_grad():
    for i, parameter in enumerate(parameters):
        parameter.fill_(float(i + 1))

shapes = [parameter.shape for parameter in parameters]
other_worker = torch.cat([
    torch.full_like(parameter, float(i + 3)).flatten()
    for i, parameter in enumerate(parameters)
])
local_worker = torch.cat([
    parameter.detach().flatten() for parameter in parameters
])


def simulate_all_reduce(packed, *args, **kwargs):
    operation = kwargs.get("op", args[0] if args else dist.ReduceOp.SUM)
    assert operation == dist.ReduceOp.SUM
    assert packed.ndim == 1 and packed.numel() == other_worker.numel()
    assert torch.allclose(packed, local_worker)
    packed.add_(other_worker)  # Simulate a SUM from two workers.


with patch.object(dist, "all_reduce", side_effect=simulate_all_reduce) as collective:
    average_model(model, world_size=2)
    assert collective.call_count == 1  # All parameters must be packed together.
    assert all(
        parameter.shape == shape
        for parameter, shape in zip(model.parameters(), shapes)
    )
    assert all(
        torch.allclose(parameter, torch.full_like(parameter, float(i + 2)))
        for i, parameter in enumerate(model.parameters())
    )
    assert all(
        current is original
        for current, original in zip(model.parameters(), parameters)
    )  # Preserve parameter objects used by the optimizer.

    average_model(model, world_size=1)
    assert collective.call_count == 1  # A single worker needs no communication.
    assert all(
        torch.allclose(parameter, torch.full_like(parameter, float(i + 2)))
        for i, parameter in enumerate(model.parameters())
    )

## Part B: Provided experiment and plotting code

These cells run training with 1, 2, and 4 workers and make the plots. They contain no student TODOs.

### Run the experiments


In [ ]:
# PROVIDED CODE — Run the 1-, 2-, and 4-worker experiments
from datetime import datetime
from pathlib import Path
import os
import platform
import socket
import subprocess
import sys

ROOT = Path.cwd().resolve()
SCRIPT = ROOT / "train_distributed.py"
assert SCRIPT.is_file() and (ROOT / "features.csv").is_file()

def launch(workers, *script_args, timeout):
    # Pick an available IPv4 port for each torchrun invocation.
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        port = sock.getsockname()[1]
    env = os.environ.copy()
    env["OMP_NUM_THREADS"] = "1"
    env["POLARS_MAX_THREADS"] = "1"
    if platform.system() == "Darwin":
        env.setdefault("GLOO_SOCKET_IFNAME", "lo0")
    if platform.system() == "Windows":
        env["USE_LIBUV"] = "0"  # Some Windows PyTorch builds lack libuv.
    if platform.system() == "Windows":
        # This avoids torchrun's static TCP store, which can require libuv on Windows.
        env.update(MASTER_ADDR="127.0.0.1", MASTER_PORT=str(port),
                   WORLD_SIZE=str(workers))
        processes = []
        try:
            for rank in range(workers):
                worker_env = {**env, "RANK": str(rank), "LOCAL_RANK": str(rank)}
                processes.append(subprocess.Popen(
                    [sys.executable, str(SCRIPT), *map(str, script_args)],
                    cwd=ROOT, env=worker_env))
            for process in processes:
                process.wait(timeout=timeout)
                if process.returncode:
                    raise subprocess.CalledProcessError(process.returncode, process.args)
        finally:
            for process in processes:
                if process.poll() is None:
                    process.kill()
        return
    command = [sys.executable, "-m", "torch.distributed.run", "--nnodes=1",
               f"--nproc-per-node={workers}", "--rdzv-backend=static",
               "--master-addr=127.0.0.1", f"--master-port={port}",
               str(SCRIPT), *map(str, script_args)]
    subprocess.run(command, cwd=ROOT, env=env, check=True, timeout=timeout)



In [ ]:
EPOCHS = 30
RUN_ROOT = ROOT / f"results_{datetime.now():%Y%m%d_%H%M%S_%f}"
for workers in (1, 2, 4):
    print(f"Running {workers} worker(s)...", flush=True)
    launch(workers, "--epochs", EPOCHS, "--output", RUN_ROOT / f"w{workers}", timeout=900)

### Plot validation loss and throughput


In [ ]:
# PROVIDED CODE — Plot training results
import csv
import math
import matplotlib.pyplot as plt

fig, (loss_ax, speed_ax) = plt.subplots(1, 2, figsize=(14, 5))
throughputs = []
counts = (1, 2, 4)
for workers in counts:
    with (RUN_ROOT / f"w{workers}" / "history.csv").open(newline="") as stream:
        rows = list(csv.DictReader(stream))
    assert len(rows) == EPOCHS and all(int(row["workers"]) == workers for row in rows)
    times = [float(row["train_time_s"]) for row in rows]
    losses = [float(row["val_loss"]) for row in rows]
    assert all(a < b for a, b in zip(times, times[1:]))
    assert all(math.isfinite(loss) and loss > 0 for loss in losses)
    assert int(rows[-1]["examples_seen"]) > 0
    loss_ax.plot(times, losses, marker="o", markersize=3, label=f"Workers = {workers}")
    throughputs.append(int(rows[-1]["examples_seen"]) / times[-1])

loss_ax.set(xlabel="Cumulative training time (s)", ylabel="Validation BCE loss")
loss_ax.legend()
speed_ax.plot(counts, throughputs, marker="o")
speed_ax.set(xlabel="Number of workers", ylabel="Training throughput (examples/s across all workers)")
speed_ax.set_xticks(counts)
fig.tight_layout()
figure_path = RUN_ROOT / "ddp_two_plots.png"
fig.savefig(figure_path, dpi=180)
plt.show()
print(f"Saved {figure_path}")


### Train with parameter mixing

In [ ]:

EPOCHS = 30
RUN_ROOT = ROOT / f"results_{datetime.now():%Y%m%d_%H%M%S_%f}"
WORKERS = 4
for k in [1,4,16]:
    print(f"Running model average every {k} iterations...", flush=True)
    launch(WORKERS, "--epochs", EPOCHS, "--output", RUN_ROOT / f"m{k}", "--average-model", k, timeout=900)

In [ ]:
import csv
import math
import matplotlib.pyplot as plt

fig, (loss_ax, speed_ax) = plt.subplots(1, 2, figsize=(14, 5))
throughputs = []
workers = 4
counts = (1, 4, 16)
for k in counts:
    with (RUN_ROOT / f"m{k}" / "history.csv").open(newline="") as stream:
        rows = list(csv.DictReader(stream))
    assert len(rows) == EPOCHS and all(int(row["workers"]) == workers for row in rows)
    times = [float(row["train_time_s"]) for row in rows]
    losses = [float(row["val_loss"]) for row in rows]
    assert all(a < b for a, b in zip(times, times[1:]))
    assert all(math.isfinite(loss) and loss > 0 for loss in losses)
    assert int(rows[-1]["examples_seen"]) > 0
    loss_ax.plot(times, losses, marker="o", markersize=3, label=f"Model Average = {k}")
    throughputs.append(int(rows[-1]["examples_seen"]) / times[-1])

loss_ax.set(xlabel="Cumulative training time (s)", ylabel="Validation BCE loss")
loss_ax.legend()
speed_ax.plot(counts, throughputs, marker="o")
speed_ax.set(xlabel="Model Average Intervals", ylabel="Training throughput (examples/s across all workers)")
speed_ax.set_xticks(counts)
fig.tight_layout()
figure_path = RUN_ROOT / "ddp_two_model_average_plot.png"
fig.savefig(figure_path, dpi=180)
plt.show()
print(f"Saved {figure_path}")